# Qwen3-4B COA / Accounting Service — Backend API

Standalone COA classification service only.

Flow: `Backend → POST /api/infer/categorize-accounting → Qwen3-4B → accounting JSON → Backend`

TDS is intentionally excluded and runs in the separate TDS service.
GST arithmetic, reconciliation, ITC and journal calculations remain in the backend.


In [ ]:
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes pydantic fastapi uvicorn nest_asyncio pyngrok requests

In [ ]:
import copy
import json
import re
import time
import datetime
import asyncio
import traceback
import threading
from typing import Optional, List, Dict, Any

import torch
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.manual_seed(0)

# Concurrency lock for safe GPU inference
inference_lock = asyncio.Lock()

# Unique Request ID generator (COA-YYYYMMDD-HHMMSS-NNNN)
_request_counter = 0
_counter_lock = threading.Lock()

def generate_coa_request_id() -> str:
    global _request_counter
    with _counter_lock:
        _request_counter += 1
        now_str = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        return f"COA-{now_str}-{_request_counter:04d}"

In [ ]:
MODEL_NAME_TEXT = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

text_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_TEXT)
text_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_TEXT,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
text_model.eval()
print("Loaded:", MODEL_NAME_TEXT)

In [ ]:
def _as_float(value):
    if value is None or value == "": return None
    try: return float(value)
    except (TypeError, ValueError): return None

def _clean_numeric(value):
    if value is None or value == "": return None
    if isinstance(value, (int, float)): return float(value)
    text = str(value).strip().replace(",", "")
    text = re.sub(r"^(?:Rs\.?|INR|₹)\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*/-\s*$", "", text)
    negative = text.startswith("(") and text.endswith(")")
    text = text.strip("()")
    try:
        number = float(text)
        return -number if negative else number
    except ValueError: return None

def safe_json_parse(text):
    if not isinstance(text, str): return None
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.IGNORECASE|re.MULTILINE)
    try: return json.loads(cleaned)
    except json.JSONDecodeError:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start >= 0 and end > start:
            try: return json.loads(cleaned[start:end+1])
            except json.JSONDecodeError: return None
        return None

def _generate_json(messages, max_new_tokens=768, model=None, tokenizer=None):
    model = model or text_model
    tokenizer = tokenizer or text_tokenizer
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors='pt').to(model.device)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=pad_id)
    generated = outputs[0][inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = safe_json_parse(raw)
    if not isinstance(parsed, dict):
        raise ValueError('Qwen3-4B did not return a valid JSON object')
    return parsed, raw

## COA / Accounting Classification

The model receives the **complete invoice JSON** plus the supplied COA. It returns one best matching supplied account for every non-empty line.

In [ ]:
def categorize_line_items(
    line_items: list,
    chart_of_accounts: list,
    vendor_name: str = None,
    invoice_context: dict = None,
    available_taxes: list = None,
    full_invoice_json: dict = None,
    req_id: str = "COA-UNKNOWN",
) -> tuple[list, str, dict, float]:
    """Focused Qwen3-4B COA/accounting classification only. Returns (normalized_results, raw_output, parsed_json, inference_latency)."""
    line_items = line_items or []
    chart_of_accounts = chart_of_accounts or []
    invoice_context = invoice_context or {}

    if isinstance(full_invoice_json, dict):
        complete_invoice = copy.deepcopy(full_invoice_json)
    else:
        complete_invoice = copy.deepcopy(invoice_context)
        complete_invoice["vendor_name"] = vendor_name or complete_invoice.get("vendor_name")
        complete_invoice["line_items"] = copy.deepcopy(line_items)

    if not isinstance(complete_invoice.get("line_items"), list):
        complete_invoice["line_items"] = copy.deepcopy(line_items)

    coa_lines = []
    accounts_by_id = {}
    accounts_by_name = {}
    for idx, acc in enumerate(chart_of_accounts, 1):
        if not isinstance(acc, dict):
            continue
        acc_id = str(acc.get("account_id") or acc.get("id") or f"ACC_{idx}").strip()
        acc_name = str(acc.get("account_name") or acc.get("name") or "").strip()
        acc_type = str(acc.get("account_type") or acc.get("type") or "expense").strip()
        if not acc_name:
            continue
        coa_lines.append({"account_id": acc_id, "account_name": acc_name, "account_type": acc_type})
        accounts_by_id[acc_id] = acc_name
        accounts_by_name[acc_name.lower()] = (acc_id, acc_name)

    system_prompt = """
You are the COA/accounting-classification component of an Indian Accounts Payable invoice pipeline.

You receive the COMPLETE invoice JSON and a supplied Chart of Accounts.
Read the entire invoice JSON before deciding. Use the full context, not just the line description.

YOUR ONLY RESPONSIBILITY:
Classify every non-empty invoice line to the single best matching account from the supplied COA.

RULES
1. Return exactly one account for every non-empty input line.
2. account_id MUST come from the supplied COA.
3. account_name MUST come from the supplied COA.
4. Never invent, rename, merge, or create an account.
5. Use description, HSN/SAC, vendor/customer context, payment context, and the complete invoice JSON.
6. GST, CGST, SGST, IGST, TDS, and other taxes are not the underlying expense account.
7. Give a confidence score from 0.0 to 1.0.
8. Set ai_needs_review=true when the best match is genuinely uncertain.
9. Give concise accounting reasoning.
10. Preserve the input line order.

STRICTLY DO NOT:
- calculate GST
- calculate TDS
- calculate invoice totals
- perform any reconciliation or arithmetic validation
- calculate journals
- balance debits and credits

The monetary and tax values in the invoice JSON are input facts. Do not rewrite or recalculate them.

OUTPUT:
Return ONLY valid JSON:
{
  "accounting": [
    {
      "line_index": 1,
      "source_description": "...",
      "account_id": "...",
      "account_name": "...",
      "confidence_score": 0.97,
      "ai_needs_review": false,
      "accounting_reason": "..."
    }
  ]
}
"""
    user_payload = {
        "complete_invoice_json": complete_invoice,
        "supplied_chart_of_accounts": coa_lines,
    }
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Classify every invoice line using the supplied COA. Return JSON only.\n\n" + json.dumps(user_payload, ensure_ascii=False, indent=2)},
    ]

    # Section 3: Model Input Logging
    print("\n" + "=" * 70)
    print("🟡 QWEN INPUT")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Model: {MODEL_NAME_TEXT}")
    print(f"Number of invoice line items: {len(line_items)}")
    print(f"Number of COA accounts: {len(coa_lines)}")
    print(f"Number of available taxes: {len(available_taxes or [])}")
    print(f"Vendor Context: {complete_invoice.get('vendor_name') or 'N/A'}")
    print(f"Invoice Subtotal: {complete_invoice.get('subtotal')} | Total: {complete_invoice.get('total_amount')}")

    # Section 4: Model Status (Started)
    infer_start_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    infer_start_t = time.time()
    print("\n" + "=" * 70)
    print("🔵 QWEN3-4B INFERENCE STARTED")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Model: {MODEL_NAME_TEXT}")
    print(f"Start time: {infer_start_dt}")
    print("🔵 STATUS: QWEN3-4B INFERENCE RUNNING")

    try:
        parsed, raw_output = _generate_json(messages, max_new_tokens=max(768, 256 * max(1, len(line_items))))
        infer_end_t = time.time()
        infer_latency = round(infer_end_t - infer_start_t, 2)
        infer_end_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Section 4: Model Status (Completed)
        print("\n" + "=" * 70)
        print("🟢 QWEN3-4B INFERENCE COMPLETED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(f"End time: {infer_end_dt}")
        print(f"Inference latency: {infer_latency}s")
        print("🟡 STATUS: QWEN3-4B RESPONSE RECEIVED")

        # Section 5: Raw Model Output
        print("\n" + "=" * 70)
        print("📥 QWEN3-4B RAW OUTPUT")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(raw_output)

        # Section 6: Parsed Output
        print("\n" + "=" * 70)
        print("🟡 PARSED COA RESULT")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
        print("🟡 STATUS: PARSING COA RESPONSE")

        model_results = parsed.get("accounting")
        if not isinstance(model_results, list):
            raise ValueError("Qwen3-4B COA output must contain an accounting array")
        normalized = []
        for pos, item in enumerate(line_items, 1):
            item = item if isinstance(item, dict) else {}
            mr = model_results[pos-1] if pos-1 < len(model_results) and isinstance(model_results[pos-1], dict) else {}
            desc = str(item.get("description") or item.get("product_name") or item.get("name") or mr.get("source_description") or "").strip()
            raw_id = str(mr.get("account_id") or "").strip()
            raw_name = str(mr.get("account_name") or "").strip()
            if raw_id in accounts_by_id:
                valid_id, valid_name = raw_id, accounts_by_id[raw_id]
            elif raw_name.lower() in accounts_by_name:
                valid_id, valid_name = accounts_by_name[raw_name.lower()]
            else:
                valid_id, valid_name = None, None
            try: confidence = float(mr.get("confidence_score", 0.0))
            except (TypeError, ValueError): confidence = 0.0
            confidence = max(0.0, min(1.0, confidence))
            normalized.append({
                "line_index": pos,
                "source_description": desc,
                "account_id": valid_id,
                "account_name": valid_name,
                "confidence_score": round(confidence, 2),
                "ai_needs_review": bool(mr.get("ai_needs_review", False) or valid_id is None),
                "accounting_reason": str(mr.get("accounting_reason") or "").strip(),
            })
        if len(model_results) != len(line_items):
            for row in normalized: row["ai_needs_review"] = True
        print("🟡 STATUS: COA RESPONSE VALIDATED")
        return normalized, raw_output, parsed, infer_latency

    except Exception as exc:
        infer_latency = round(time.time() - infer_start_t, 2)
        fallback = [{
            "line_index": pos,
            "source_description": (item.get("description") if isinstance(item, dict) else "") or "",
            "account_id": None, "account_name": None, "confidence_score": 0.0,
            "ai_needs_review": True, "accounting_reason": f"stage3_coa_error:{type(exc).__name__}: {exc}",
        } for pos, item in enumerate(line_items, 1)]
        return fallback, f"ERROR: {str(exc)}", {"error": str(exc), "accounting": fallback}, infer_latency

## Backend API Contract & Complete Lifecycle Logging

Endpoint: `POST /api/infer/categorize-accounting`

Accepts:
```json
{
  "invoice_json": { ... },
  "chart_of_accounts": [ ... ],
  "available_taxes": [ ... ]
}
```

Returns:
```json
{
  "accounting": [ ... ]
}
```

In [ ]:
from fastapi import FastAPI, HTTPException
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI(title='Qwen3-4B Accounting / COA API')

class AccountingRequest(BaseModel):
    invoice_json: Dict[str, Any]
    chart_of_accounts: List[Dict[str, Any]] = Field(default_factory=list)
    available_taxes: List[Dict[str, Any]] = Field(default_factory=list)

@app.get('/health')
async def health():
    return {'status': 'ok', 'service': 'qwen3-4b-coa'}

@app.post('/api/infer/categorize-accounting')
async def categorize_accounting_endpoint(req: AccountingRequest):
    req_id = generate_coa_request_id()
    req_start_t = time.time()
    now_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_stage = "REQUEST_RECEIPT"

    # Section 1: Request Status (Received)
    print("\n" + "=" * 70)
    print("🟢 COA REQUEST RECEIVED")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Time: {now_dt}")
    print("Method: POST")
    print("Path: /api/infer/categorize-accounting")
    print("Status: RECEIVED")

    try:
        # Section 2: Request Details
        print("\n🟡 STATUS: VALIDATING REQUEST")
        current_stage = "VALIDATING_REQUEST"
        if not isinstance(req.invoice_json, dict):
            raise ValueError("invoice_json must be a valid JSON object")

        print("🟡 STATUS: REQUEST VALIDATED")
        print("\n---------------------- REQUEST RECEIVED ----------------------")
        print(f"Request ID: {req_id}")
        print(f"Timestamp: {now_dt}")
        print("HTTP Method: POST")
        print("Endpoint: /api/infer/categorize-accounting")

        print("\n---------------------- INVOICE JSON ----------------------")
        print(json.dumps(req.invoice_json, ensure_ascii=False, indent=2))

        print("\n---------------------- CHART OF ACCOUNTS -----------------")
        print(f"Count: {len(req.chart_of_accounts)}")
        print(json.dumps(req.chart_of_accounts, ensure_ascii=False, indent=2))

        print("\n---------------------- AVAILABLE TAXES -------------------")
        print(f"Count: {len(req.available_taxes)}")
        print(json.dumps(req.available_taxes, ensure_ascii=False, indent=2))

        print("\n🟡 STATUS: PREPARING QWEN INPUT")
        current_stage = "PREPARING_QWEN_INPUT"
        line_items = req.invoice_json.get('line_items') or []

        # Section 10: GPU Queue Status
        if inference_lock.locked():
            print("\n🟠 STATUS: WAITING FOR GPU / INFERENCE LOCK")
        else:
            print("\n🟡 STATUS: WAITING FOR QWEN3-4B")

        async with inference_lock:
            print("🔵 STATUS: GPU AVAILABLE — STARTING INFERENCE")
            current_stage = "INFERENCE_EXECUTION"
            accounting, raw_output, parsed_json, infer_latency = categorize_line_items(
                line_items=line_items,
                chart_of_accounts=req.chart_of_accounts,
                vendor_name=req.invoice_json.get('vendor_name') or '',
                invoice_context=req.invoice_json,
                available_taxes=req.available_taxes,
                full_invoice_json=req.invoice_json,
                req_id=req_id,
            )

        current_stage = "RESPONSE_PREPARATION"
        response_payload = {'accounting': accounting}
        total_latency = round(time.time() - req_start_t, 2)

        # Section 7: Final Response
        print("\n" + "=" * 70)
        print("📤 RESPONSE SENT TO BACKEND")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(response_payload, ensure_ascii=False, indent=2))
        print("\nStatus: SUCCESS")
        print(f"Total request latency: {total_latency} seconds")
        print("🟢 STATUS: RESPONSE SENT TO BACKEND")

        # Section 11: Final Request Summary
        print("\n" + "=" * 70)
        print("✅ COA REQUEST COMPLETED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: SUCCESS\n")
        print("Stages:")
        print("  ✅ Request received")
        print("  ✅ Request validated")
        print("  ✅ Qwen input prepared")
        print("  ✅ Qwen inference completed")
        print("  ✅ Output parsed")
        print("  ✅ COA validated")
        print("  ✅ Response sent\n")
        print(f"Invoice line items: {len(line_items)}")
        print(f"COA accounts: {len(req.chart_of_accounts)}")
        print(f"COA results: {len(accounting)}")
        print(f"Inference latency: {infer_latency}s")
        print(f"Total latency: {total_latency}s")
        print("=" * 70 + "\n")

        return response_payload

    except Exception as exc:
        total_latency = round(time.time() - req_start_t, 2)
        # Section 8: Error Status
        print("\n" + "=" * 70)
        print("🔴 COA REQUEST FAILED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: FAILED")
        print(f"Stage: {current_stage}")
        print(f"Error Type: {type(exc).__name__}")
        print(f"Error Message: {str(exc)}")
        print(f"Total Latency: {total_latency}s")
        print("\nTraceback:")
        traceback.print_exc()
        print("=" * 70 + "\n")
        raise HTTPException(status_code=500, detail=str(exc))

## Start the Continuous API Server

The server runs continuously and handles multiple sequential/concurrent backend requests without shutting down.

In [ ]:
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
public_url = "http://0.0.0.0:8000"
ngrok_status = "NOT_CONFIGURED"

if NGROK_AUTH_TOKEN:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    ngrok_status = "CONNECTED"

# Section 12: Server Startup Status
print("\n" + "=" * 70)
print("🚀 COA INFERENCE SERVER")
print("=" * 70)
print(f"Service: Qwen3-4B COA")
print(f"Status: RUNNING")
print(f"Local URL: http://0.0.0.0:8000")
print(f"Health: {public_url}/health")
print(f"Endpoint: {public_url}/api/infer/categorize-accounting")
print(f"ngrok: {ngrok_status}")
print("=" * 70)
print("\n🟢 READY — WAITING FOR BACKEND REQUESTS\n")

config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
# Run continuously to accept continuous requests
await server.serve()

## Optional Local Diagnostic Test

In [ ]:
import requests
test_payload = {
    'invoice_json': {
        'invoice_number': 'TEST-COA-001',
        'vendor_name': 'Example Cloud Vendor',
        'customer_name': 'Example Company',
        'line_items': [
            {'description':'Cloud infrastructure subscription - enterprise plan', 'quantity':1, 'unit_price':50000, 'taxable_amount':50000}
        ],
        'subtotal':50000,
        'tax_total':9000,
        'total_amount':59000
    },
    'chart_of_accounts': [
        {'account_id':'1001','account_name':'IT and Internet Expenses','account_type':'expense'},
        {'account_id':'1002','account_name':'Consulting Expense','account_type':'expense'},
        {'account_id':'1003','account_name':'Office Supplies','account_type':'expense'},
        {'account_id':'1004','account_name':'Merchandise','account_type':'expense'}
    ],
    'available_taxes': [
        {'tax_id': 'TAX_18', 'tax_name': 'GST 18%', 'tax_rate': 18.0, 'tax_type': 'GST'}
    ]
}
resp = requests.post('http://127.0.0.1:8000/api/infer/categorize-accounting', json=test_payload, timeout=120)
print('HTTP', resp.status_code)
print(json.dumps(resp.json(), ensure_ascii=False, indent=2))